In [1]:
from pathlib import Path
CONFIG = {
    'project_root': '/home/featurize/work/biological_prior_validation',
    'run_name': 'step2_q05_seed20261008',
    'seed': 20261008,
    'q': 0.5,
    'chunk_rows': 500_000,
    'cells_per_block': 5_000,
    'n_selected_genes': 2_500,  
    'min_free_gib': 20,        
    'keep_staging': False,    
}
ROOT = Path(CONFIG['project_root'])
print('Project:', ROOT)


Project: /home/featurize/work/biological_prior_validation


In [2]:
"""Step 2. Counts are split before expression-based selection; targets are sealed by convention."""
from pathlib import Path
import gzip, io, json, hashlib, shutil, time, sys, platform, subprocess
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
import pyarrow.parquet as pq

VERSION='step2-count-split-v1'
ENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest() if '__file__' in globals() else None
LINES=['A172','T98G','U87MG']
DRUGS=['Vehicle','Lapatinib','Nintedanib','Trametinib','ZSTK474']

def digest(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()

def json_save(p,value):
    p=Path(p);t=p.with_name(p.name+'.tmp')
    t.write_text(json.dumps(value,ensure_ascii=False,indent=2,default=str));t.replace(p)

class HashingReader:
    def __init__(self,f):self.f=f;self.h=hashlib.sha256();self.n=0
    def read(self,n=-1):
        b=self.f.read(n);self.h.update(b);self.n+=len(b);return b
    def seekable(self):return False

def split_values(counts,q,rng):
    c=np.asarray(counts)
    if c.dtype.kind not in 'iu' or (c<0).any():raise ValueError('Counts must be nonnegative integers')
    if (c>np.iinfo(np.int64).max).any():raise OverflowError('Counts exceed int64')
    if not np.isfinite(q) or not 0<=q<=1:raise ValueError('q must be within [0,1]')
    c=c.astype(np.int64);a=rng.binomial(c,q).astype(np.int64);b=c-a
    assert (a>=0).all() and (b>=0).all() and np.array_equal(a+b,c)
    return a,b

def normalize_A(a,library_sizes=None,scale=10000):
    a=sp.csr_matrix(a,dtype=np.float64)
    if not np.isfinite(a.data).all() or (a.data<0).any():raise ValueError('Invalid A')
    lib=np.asarray(a.sum(axis=1)).ravel() if library_sizes is None else np.asarray(library_sizes,dtype=np.float64)
    if len(lib)!=a.shape[0] or not np.isfinite(lib).all() or (lib<0).any():raise ValueError('Invalid A library sizes')
    if (np.asarray(a.sum(axis=1)).ravel()>lib+1e-8).any():raise ValueError('A library is smaller than observed selected-gene counts')
    multiplier=np.divide(scale,lib,out=np.zeros_like(lib),where=lib>0)
    y=sp.diags(multiplier).dot(a).tocsr();y.data=np.log1p(y.data);y.eliminate_zeros()
    return y,lib

def expected_B_counts(log_expression,library_A,q,scale=10000):
    """Fixed inverse log1p count-scale mapping; accepts no B or C information."""
    if not 0<q<1:raise ValueError('Calibration requires 0<q<1')
    y=np.asarray(log_expression,dtype=np.float64);lib=np.asarray(library_A,dtype=np.float64)
    if y.ndim!=2 or lib.shape!=(y.shape[0],) or not np.isfinite(y).all() or not np.isfinite(lib).all() or (lib<0).any():raise ValueError('Invalid prediction/library shape or values')
    with np.errstate(over='raise',invalid='raise'):
        return np.expm1(np.maximum(y,0))*lib[:,None]/scale*(1-q)/q

def _save_npz(path,a):
    t=Path(path).with_name(Path(path).name+'.tmp.npz');sp.save_npz(t,a,compressed=True);t.replace(path)

def _file_record(path):return dict(path=str(path),sha256=digest(path),size_bytes=Path(path).stat().st_size)

def prepare_inputs(cfg):
    root=Path(cfg['project_root']);audit=root/'data/metadata_audit';proc=root/'data/processed'
    if not 0<cfg['q']<1 or cfg['chunk_rows']<1 or cfg['cells_per_block']<1:raise ValueError('Invalid splitting configuration')
    scan=json.loads((root/'results/raw/fresh_count_scan_summary.json').read_text())
    if scan['status']!='full_scan_complete':raise ValueError('Step 1 full scan is required')
    manifest=json.loads((audit/'download_manifest.json').read_text())
    raw=root/'data/raw/GSM7056149_sciPlexGxE_2_UMI.count.matrix.gz';record=manifest['files'][raw.name]
    if record['status']!='complete' or raw.stat().st_size!=record['size_bytes'] or record['sha256']!=scan['raw_sha256']:raise ValueError('Raw source receipt mismatch')
    cell_path=proc/'fresh_retained_cell_annotations.parquet';cells=pd.read_parquet(cell_path,columns=['cell_idx','cell_barcode','total_umi','matrix_row']).sort_values('matrix_row').reset_index(drop=True)
    if len(cells)!=scan['retained_cells'] or not cells.cell_idx.is_unique or not cells.cell_barcode.is_unique or cells.cell_barcode.isna().any():raise ValueError('Cell mapping mismatch')
    if not np.array_equal(cells.matrix_row,np.arange(len(cells))) or not cells.total_umi.ge(scan['min_umi']).all():raise ValueError('Invalid retained row mapping')
    if not cells.cell_idx.between(1,scan['candidate_barcode_rows']).all():raise ValueError('Cell index out of bounds')
    metadata_path=proc/'fresh_audit_cell_metadata.parquet'
    available=set(pq.read_schema(metadata_path).names)
    wanted=['cell_barcode','cell_line','drug','replicate','dose_raw','sgRNA_raw','target_raw','control_label','hash_plate','hash_label_well','library_id','culture_batch','context_eligible']
    md=pd.read_parquet(metadata_path,columns=[c for c in wanted if c in available])
    if not {'cell_barcode','cell_line','drug','replicate','context_eligible'}<=set(md) or not md.cell_barcode.is_unique:raise ValueError('Metadata mapping missing/ambiguous')
    cells=cells.merge(md,on='cell_barcode',how='left',validate='one_to_one')
    if len(cells)!=len(md) or cells.context_eligible.isna().any():raise ValueError('Retained metadata set differs')
    cells['context_eligible']=cells.context_eligible.astype(bool)
    genes_path=root/'data/raw/GSM7056149_sciPlexGxE_2_gene.annotations.txt.gz'
    if digest(genes_path)!=manifest['files'][genes_path.name]['sha256']:raise ValueError('Raw gene annotation changed')
    genes=pd.read_csv(genes_path,sep='\t',header=None,dtype='string',keep_default_na=False)
    if genes.shape[1]!=2 or len(genes)!=scan['gene_annotation_rows']:raise ValueError('Gene annotation schema/size mismatch')
    genes.columns=['gene_id','gene_symbol'];genes.gene_symbol=genes.gene_symbol.str.strip().replace('',pd.NA);genes.insert(0,'gene_idx',np.arange(1,len(genes)+1));genes['gene_col']=genes.gene_idx-1
    source=dict(version=VERSION,engine_sha256=ENGINE_SHA,raw_sha256=record['sha256'],raw_size=raw.stat().st_size,raw_mtime_ns=raw.stat().st_mtime_ns,
                cells_sha256=digest(cell_path),metadata_sha256=digest(metadata_path),genes_sha256=digest(genes_path),
                numpy=np.__version__,scipy=scipy.__version__,q=cfg['q'],seed=cfg['seed'],chunk_rows=cfg['chunk_rows'],cells_per_block=cfg['cells_per_block'])
    return raw,scan,cells,genes,source

def split_raw(cfg):
    """Restartable stream pass + bounded-row sparse shards. No gene selection occurs here."""
    if Path(cfg['run_name']).name!=cfg['run_name'] or cfg['run_name'] in ['', '.', '..']:raise ValueError('run_name must be one folder name')
    raw,scan,cells,genes,identity=prepare_inputs(cfg)
    root=Path(cfg['project_root']);run=root/'data/splits'/cfg['run_name'];run.mkdir(parents=True,exist_ok=True)
    identity_file=run/'identity.json'
    if identity_file.exists():
        if json.loads(identity_file.read_text())!=identity:raise ValueError('Existing run identity differs; use a new run_name')
    elif any(run.iterdir()):raise ValueError('Existing untracked run folder; use a new run_name')
    else:json_save(identity_file,identity)
    n=len(cells);ng=len(genes);block_size=cfg['cells_per_block'];nblocks=(n+block_size-1)//block_size
    cells['row_block']=cells.matrix_row//block_size;cells['local_row']=cells.matrix_row%block_size
    cells.to_parquet(run/'cells.parquet',index=False);genes.to_parquet(run/'genes_all.parquet',index=False)
    checkpoint_file=run/'stream_checkpoint.json';stage=run/'staging';stage.mkdir(exist_ok=True)
    rng=np.random.Generator(np.random.PCG64(cfg['seed']))
    checkpoint=json.loads(checkpoint_file.read_text()) if checkpoint_file.exists() else dict(last_chunk=-1,phase='streaming',rng_state=rng.bit_generator.state)
    if checkpoint['phase']=='complete':
        result=json.loads((run/'split_summary.json').read_text())
        if result['source_identity']!=identity:raise ValueError('Completed split summary identity mismatch')
        for block in range(nblocks):
            folder=run/f'block_{block:05d}';receipt=json.loads((folder/'receipt.json').read_text())
            for item in receipt['files']:
                if digest(folder/item['name'])!=item['sha256']:raise ValueError('Completed count shard changed')
        print('Reuse checksum-verified completed split; no raw reread')
        return run,result
    if shutil.disk_usage(run).free<cfg.get('min_free_gib',0)*1024**3:raise OSError('Free disk below configured start threshold; this threshold is not a size guarantee')
    if checkpoint['phase']=='streaming':
        rng.bit_generator.state=checkpoint['rng_state']
        lookup=np.full(scan['candidate_barcode_rows']+1,-1,dtype=np.int32)
        lookup[cells.cell_idx.to_numpy(dtype=np.int64)]=cells.matrix_row.to_numpy(dtype=np.int32)
        rows_seen=0;umi_seen=0;started=time.monotonic()
        with raw.open('rb') as f:
            h=HashingReader(f)
            with gzip.GzipFile(fileobj=h,mode='rb') as gz,io.TextIOWrapper(gz) as text:
                for chunk_id,d in enumerate(pd.read_csv(text,sep=r'\s+',header=None,dtype=np.int64,chunksize=cfg['chunk_rows'])):
                    if d.shape[1]!=3 or d.isna().any().any():raise ValueError('Expected three integer count columns')
                    d.columns=['gene','cell','C']
                    if not d.gene.between(1,ng).all() or not d.cell.between(1,scan['candidate_barcode_rows']).all() or not d.C.ge(0).all():raise ValueError('Invalid raw coordinates/counts')
                    if int(d.C.max())>np.iinfo(np.int64).max//len(d):raise OverflowError('Unsafe int64 chunk sum')
                    rows_seen+=len(d);umi_seen+=int(d.C.sum())
                    if chunk_id<=checkpoint['last_chunk']:continue
                    mapped=lookup[d.cell.to_numpy()];keep=(mapped>=0)&d.C.gt(0).to_numpy()
                    counts=d.C.to_numpy()[keep];a,_=split_values(counts,cfg['q'],rng)
                    selected=pd.DataFrame(dict(row=mapped[keep],col=d.gene.to_numpy()[keep]-1,C=counts,A=a))
                    selected['block']=selected.row//block_size
                    final=stage/f'chunk_{chunk_id:06d}';temp=stage/f'chunk_{chunk_id:06d}.tmp'
                    # Only uncheckpointed temporary files owned by this run can be replaced.
                    if temp.exists():shutil.rmtree(temp)
                    if final.exists():shutil.rmtree(final)
                    temp.mkdir();records=[]
                    for block,part in selected.groupby('block',sort=True):
                        path=temp/f'block_{int(block):05d}.parquet';part.drop(columns='block').to_parquet(path,index=False,compression='zstd')
                        records.append(dict(name=path.name,sha256=digest(path),rows=len(part)))
                    json_save(temp/'receipt.json',dict(chunk=chunk_id,files=records));temp.replace(final)
                    checkpoint=dict(last_chunk=chunk_id,phase='streaming',rng_state=rng.bit_generator.state)
                    json_save(checkpoint_file,checkpoint)
                    if chunk_id%20==0:print(f'Raw pass: {rows_seen:,} records, {(time.monotonic()-started)/60:.1f} min',flush=True)
            if h.n!=raw.stat().st_size or h.h.hexdigest()!=scan['raw_sha256']:raise ValueError('Full raw SHA/size verification failed')
        if rows_seen!=scan['coordinate_rows'] or umi_seen!=scan['total_umi']:raise ValueError('Full scan totals differ from Step 1')
        checkpoint.update(phase='building',raw_rows=rows_seen,raw_umi=umi_seen,raw_sha256=scan['raw_sha256']);json_save(checkpoint_file,checkpoint)
        del lookup
    receipts=[stage/f'chunk_{i:06d}/receipt.json' for i in range(checkpoint['last_chunk']+1)]
    if any(not p.exists() for p in receipts):raise ValueError('Committed stream chunk missing')
    by_block={i:[] for i in range(nblocks)}
    for receipt in receipts:
        for record in json.loads(receipt.read_text())['files']:
            path=receipt.parent/record['name'];block=int(path.stem.split('_')[1])
            if digest(path)!=record['sha256']:raise ValueError('Staged split chunk checksum mismatch')
            by_block[block].append(path)
    totals=[]
    for block in range(nblocks):
        start=block*block_size;stop=min(start+block_size,n);folder=run/f'block_{block:05d}';folder.mkdir(exist_ok=True)
        done=folder/'receipt.json'
        if done.exists():
            r=json.loads(done.read_text())
            for record in r['files']:
                if digest(folder/record['name'])!=record['sha256']:raise ValueError('Completed count shard changed')
            totals.append(r['summary']);continue
        # One bounded cell block is materialized; no complete dense matrix is constructed.
        paths=by_block[block]
        frame=pd.concat([pd.read_parquet(p) for p in paths],ignore_index=True) if paths else pd.DataFrame(columns=['row','col','C','A'])
        row=frame.row.to_numpy(dtype=np.int64)-start;col=frame.col.to_numpy(dtype=np.int64)
        c=sp.coo_matrix((frame.C.to_numpy(dtype=np.int64),(row,col)),shape=(stop-start,ng)).tocsr();c.sum_duplicates();c.eliminate_zeros()
        a=sp.coo_matrix((frame.A.to_numpy(dtype=np.int64),(row,col)),shape=c.shape).tocsr();a.sum_duplicates();a.eliminate_zeros();b=(c-a).tocsr();b.eliminate_zeros()
        if (b.data<0).any() or (a+b-c).nnz:raise AssertionError('Elementwise A+B=C failure')
        lib_c=np.asarray(c.sum(axis=1)).ravel();lib_a=np.asarray(a.sum(axis=1)).ravel();lib_b=np.asarray(b.sum(axis=1)).ravel()
        if not np.array_equal(lib_c,cells.total_umi.iloc[start:stop].to_numpy(dtype=np.int64)):raise AssertionError('All-gene retained UMI differs from Step 1')
        if not np.array_equal(lib_a+lib_b,lib_c):raise AssertionError('Library conservation failure')
        files=[]
        for name,matrix in [('C_counts.npz',c),('A_input.npz',a),('B_target.npz',b)]:
            _save_npz(folder/name,matrix);loaded=sp.load_npz(folder/name)
            if (loaded-matrix).nnz:raise AssertionError('Sparse save/load failed')
            files.append(dict(name=name,sha256=digest(folder/name)))
        libs=pd.DataFrame(dict(matrix_row=np.arange(start,stop),library_C=lib_c,library_A=lib_a,library_B=lib_b,A_zero_library=lib_a==0));libs.to_parquet(folder/'libraries.parquet',index=False)
        files.append(dict(name='libraries.parquet',sha256=digest(folder/'libraries.parquet')))
        summary=dict(block=block,rows=stop-start,C_umi=int(lib_c.sum()),A_umi=int(lib_a.sum()),B_umi=int(lib_b.sum()),C_nnz=c.nnz,A_nnz=a.nnz,B_nnz=b.nnz,A_zero_cells=int((lib_a==0).sum()))
        json_save(done,dict(summary=summary,files=files));totals.append(summary)
        del frame,c,a,b,loaded
        print(f'Count shards: {block+1}/{nblocks}',flush=True)
    totals=pd.DataFrame(totals);totals.to_csv(run/'block_summary.csv',index=False)
    assert int(totals.A_umi.sum())+int(totals.B_umi.sum())==int(totals.C_umi.sum())==int(cells.total_umi.sum())
    result=dict(status='complete',run_path=str(run),seed=cfg['seed'],q=cfg['q'],algorithm='PCG64 / NumPy binomial per positive retained coordinate record',
                numpy=np.__version__,n_cells=n,n_genes=ng,n_blocks=nblocks,A_umi=int(totals.A_umi.sum()),B_umi=int(totals.B_umi.sum()),C_umi=int(totals.C_umi.sum()),
                actual_A_fraction=float(totals.A_umi.sum()/totals.C_umi.sum()) if totals.C_umi.sum() else None,A_zero_cells=int(totals.A_zero_cells.sum()),
                equality='elementwise and all-gene library A+B=C checked',raw_sha256=scan['raw_sha256'],raw_crc='full_stream_checked',
                independence='molecular_holdout; biological independence unresolved; unconditional A/B independence requires distribution assumptions',
                population_selection='Fixed Step 1 C>=500 and author-metadata/QC population; this conditions on C=A+B',
                source_identity=identity)
    json_save(run/'split_summary.json',result);checkpoint['phase']='complete';json_save(checkpoint_file,checkpoint)
    if not cfg.get('keep_staging',False):shutil.rmtree(stage)
    return run,result

def stats_A(run,eligible):
    """Only A_input.npz is opened. B/C/library_C/library_B are never used."""
    run=Path(run);genes=pd.read_parquet(run/'genes_all.parquet');ng=len(genes)
    sums=np.zeros(ng);squares=np.zeros(ng);detected=np.zeros(ng,dtype=np.int64);n_fit=0
    for folder in sorted(run.glob('block_[0-9]*')):
        a=sp.load_npz(folder/'A_input.npz').tocsr();block=int(folder.name.split('_')[1]);start=block*json.loads((run/'identity.json').read_text())['cells_per_block']
        keep=np.asarray(eligible[start:start+a.shape[0]],dtype=bool);sub=a[keep]
        lib=np.asarray(sub.sum(axis=1)).ravel();positive=lib>0;sub=sub[positive];y,_=normalize_A(sub)
        n_fit+=y.shape[0];sums+=np.asarray(y.sum(axis=0)).ravel();squares+=np.asarray(y.multiply(y).sum(axis=0)).ravel();detected+=np.asarray((sub>0).sum(axis=0)).ravel()
    if n_fit<2:raise ValueError('At least two eligible positive-A-library cells required for gene selection')
    genes['A_logcp10k_mean']=sums/n_fit;genes['A_logcp10k_variance']=np.maximum((squares-sums*sums/n_fit)/(n_fit-1),0);genes['A_detection_fraction']=detected/n_fit
    return genes,n_fit

def preprocess_A(cfg,run):
    root=Path(cfg['project_root']);run=Path(run);out=run/'A_preprocessing';out.mkdir(exist_ok=True)
    preprocess_identity=dict(engine_sha256=ENGINE_SHA,run_identity_sha256=digest(run/'identity.json'),selected_gene_count=cfg['n_selected_genes'],ppi_mapping_sha256=digest(root/'data/metadata_audit/ppi_gene_mapping.csv'),ppi_edges_sha256=digest(root/'data/processed/ppi_annotation_reference_edges.parquet'))
    lock=out/'preprocessing_identity.json'
    if lock.exists() and json.loads(lock.read_text())!=preprocess_identity:raise ValueError('Frozen preprocessing definition changed; use a new run_name, not final B to select parameters')
    json_save(lock,preprocess_identity)
    cells=pd.read_parquet(run/'cells.parquet');eligible=cells.context_eligible.to_numpy(dtype=bool)
    stats,n_fit=stats_A(run,eligible)
    mapping=pd.read_csv(root/'data/metadata_audit/ppi_gene_mapping.csv',dtype={'gene_id':'string','gene_symbol':'string'},keep_default_na=False,na_values=[''])
    if mapping.gene_idx.duplicated().any():raise ValueError('PPI gene mapping gene_idx is ambiguous')
    raw=pd.read_parquet(run/'genes_all.parquet');joined=raw[['gene_idx','gene_id','gene_symbol']].merge(mapping[['gene_idx','gene_id','gene_symbol','ppi_edge_supported']],on='gene_idx',validate='one_to_one',suffixes=('','_ppi'))
    if len(joined)!=len(raw) or not joined.gene_id.fillna('').eq(joined.gene_id_ppi.fillna('')).all() or not joined.gene_symbol.fillna('').eq(joined.gene_symbol_ppi.fillna('')).all():raise ValueError('PPI mapping identifiers differ from raw annotation')
    support=joined.set_index('gene_idx').ppi_edge_supported
    stats['ppi_supported']=stats.gene_idx.map(support).eq(True)
    candidates=stats.loc[stats.ppi_supported & stats.gene_symbol.notna() & stats.A_logcp10k_variance.gt(0)]
    selected=candidates.sort_values(['A_logcp10k_variance','A_detection_fraction','gene_idx'],ascending=[False,False,True],kind='stable').drop_duplicates('gene_symbol').head(cfg['n_selected_genes']).copy()
    if selected.empty:raise ValueError('No A-variable PPI-supported genes; cannot preprocess silently')
    selected['selected_col']=np.arange(len(selected));cols=selected.gene_col.to_numpy(dtype=np.int64)
    stats.to_parquet(out/'gene_stats_A.parquet',index=False);selected.to_parquet(out/'selected_genes_A.parquet',index=False);selected.to_csv(out/'selected_genes_A.csv',index=False)
    files=[];evaluation_mask=np.zeros(len(cells),dtype=bool)
    for folder in sorted(run.glob('block_[0-9]*')):
        a=sp.load_npz(folder/'A_input.npz');library=np.asarray(a.sum(axis=1)).ravel()
        start=int(folder.name.split('_')[1])*json.loads((run/'identity.json').read_text())['cells_per_block']
        evaluation_mask[start:start+len(library)]=eligible[start:start+len(library)] & (library>0)
        y,_=normalize_A(a[:,cols],library_sizes=library)
        p=out/(folder.name+'_Y_A.npz');_save_npz(p,y);files.append(_file_record(p))
        pd.DataFrame({'local_row':np.arange(len(library)),'library_A':library,'A_zero_library':library==0}).to_parquet(out/(folder.name+'_A_libraries.parquet'),index=False)
    cells=cells.drop(columns=['total_umi'],errors='ignore');cells['evaluation_eligible']=evaluation_mask;cells.to_parquet(out/'evaluation_cells.parquet',index=False)
    json_save(out/'evaluation_pair_contract.json',dict(cells_file='evaluation_cells.parquet',cells_sha256=digest(out/'evaluation_cells.parquet'),genes_file='selected_genes_A.parquet',genes_sha256=digest(out/'selected_genes_A.parquet'),n_cells=int(evaluation_mask.sum()),n_genes=len(selected),n_pairs=int(evaluation_mask.sum())*len(selected),pair_rule='all selected genes, including observed zeros, for context-eligible positive-A-library cells; frozen without B access',q=cfg['q'],normalization_scale=10000,method_comparison='all methods must use exactly these same cell-gene pairs'))
    ppi=pd.read_parquet(root/'data/processed/ppi_annotation_reference_edges.parquet');symbols=set(selected.gene_symbol)
    edges=ppi.loc[ppi.gene_a.isin(symbols)&ppi.gene_b.isin(symbols)].copy();edges.to_parquet(out/'selected_PPI_edges.parquet',index=False)
    result=dict(status='complete',fitting_scope='all context-eligible positive-library A cells within-cell molecular holdout',n_fit_cells=n_fit,
                eligible_cells=int(eligible.sum()),evaluation_cells=int(evaluation_mask.sum()),selected_genes=len(selected),requested_genes=cfg['n_selected_genes'],selected_PPI_edges=len(edges),
                gene_rule='A-only logCP10K sample variance, detection fraction, gene_idx deterministic tie; unique symbols',normalization='log1p(10000*A/all-gene-A-library)',
                zero_A_library='retained as zero input; excluded from gene-stat fitting; no B/C rescue',B_access='none',
                graph_metacell_model='not fitted in Step 2; future construction/fitting/updates must use A only',
                cross_replicate='not development-only: do not reuse this within-cell selection for cross-replicate confirmation',files=files)
    json_save(out/'preprocessing_summary.json',result)
    return result

def save_protocol(cfg,run,split_summary,prep_summary):
    root=Path(cfg['project_root']);run=Path(run)
    policy=[('cell_population','Step 1 C>=500 and author QC fixed','selection conditioning explicitly recorded'),('normalization/gene_selection','A only','implemented'),('metacell/cell_graph/estimator/adaptive_updates','A only','future fitting not performed here'),('B aggregation','fixed A-only assignment','never regroup using B'),('hyperparameter_selection','pre-fixed settings or separate development targets','final B excluded'),('final_evaluation','B only after freeze','no B-based preprocessing/calibration'),('count_calibration','A library and q only','fixed expected_B_counts function'),('cross_replicate_sensitivity','C unsplit; dev-only preprocessing in each direction','batch/label sensitivity; independent biological status unresolved')]
    pd.DataFrame(policy,columns=['stage','allowed_information','status']).to_csv(run/'information_flow.csv',index=False)
    directions=[dict(development_replicate=a,evaluation_replicate=b,count_input='C_counts.npz',count_splitting=False,preprocessing='fit on development C only; re-fit genes/normalization rules per direction',interpretation='cross-label sensitivity, not proven independent biological replication',execution='saved_branch_NOT_FITTED_YET') for a,b in [('replicate_1','replicate_2'),('replicate_2','replicate_1')]]
    json_save(run/'unsplit_cross_replicate_sensitivity.json',directions)
    cells=pd.read_parquet(run/'cells.parquet');good=cells[cells.context_eligible]
    fields=[f for f in ['cell_line','drug','replicate','dose_raw','sgRNA_raw','target_raw','hash_plate','hash_label_well','library_id'] if f in good]
    counts=good.groupby(fields,dropna=False,observed=True).size().rename('n_cells').reset_index();assert int(counts.n_cells.sum())==len(good)
    counts.to_csv(run/'context_counts.csv',index=False)
    universe=pd.MultiIndex.from_product([LINES,DRUGS],names=['cell_line','drug']);totals=good.groupby(['cell_line','drug']).size().reindex(universe,fill_value=0).rename('n_cells').reset_index();totals.to_csv(run/'context_totals.csv',index=False)
    json_save(run/'step2_config.json',cfg)
    try:commit=subprocess.check_output(['git','-C',str(root),'rev-parse','HEAD'],stderr=subprocess.DEVNULL,text=True).strip()
    except (OSError,subprocess.CalledProcessError):commit='not_available'
    json_save(run/'environment.json',dict(python=sys.version,numpy=np.__version__,scipy=scipy.__version__,pandas=pd.__version__,platform=platform.platform(),git_commit=commit,seed=cfg['seed']))
    (root/'results/raw').mkdir(parents=True,exist_ok=True)
    json_save(root/'results/raw/step2_summary.json',dict(split=split_summary,preprocessing=prep_summary,
                methods='count-splitting data and A-only preprocessing complete; estimator fitting/scoring/adaptive updates not run',
                sensitivity='unsplit C-only pooled-proportion reference baseline computed in both directions; not a method comparison or independent biological validation'))
    (root/'configs').mkdir(exist_ok=True);json_save(root/'configs/step2_count_split.json',cfg)
    return totals

def unsplit_sensitivity(run):
    """Numerical, exploratory C-only reference baseline; no A/B accessed.

    Predict each cell-line/drug/dose stratum from the other replicate's pooled
    gene proportions, using a fixed unit Dirichlet pseudocount. Evaluate all raw
    genes. Per-cell C library is observed for scaling: observational sensitivity,
    not molecular holdout or proof of independent biological generalization.
    sgRNA is not fitted as a covariate in this deliberately simple reference.
    """
    run=Path(run);cells=pd.read_parquet(run/'cells.parquet');identity=json.loads((run/'identity.json').read_text())
    ng=len(pd.read_parquet(run/'genes_all.parquet'));size=identity['cells_per_block']
    good=cells.context_eligible & cells.replicate.isin(['replicate_1','replicate_2'])
    work=cells.loc[good,['cell_line','drug','dose_raw','replicate']].copy()
    work['dose_label']=work.dose_raw.astype('string').fillna('<missing>')
    keys=['cell_line','drug','dose_label','replicate']
    groups=work[keys].drop_duplicates().sort_values(keys).reset_index(drop=True)
    gid=dict(zip(map(tuple,groups[keys].values),range(len(groups))))
    labels=np.full(len(cells),-1,dtype=np.int32)
    labels[work.index]=[gid[tuple(x)] for x in work[keys].values]
    pooled=np.zeros((len(groups),ng),dtype=np.float64);n_cells=np.zeros(len(groups),dtype=np.int64)
    for folder in sorted(run.glob('block_[0-9]*')):
        c=sp.load_npz(folder/'C_counts.npz').tocsr();start=int(folder.name.split('_')[1])*size;idx=labels[start:start+c.shape[0]];valid=idx>=0
        assignment=sp.csr_matrix((np.ones(valid.sum()),(idx[valid],np.flatnonzero(valid))),shape=(len(groups),c.shape[0]))
        pooled+= (assignment@c).toarray();n_cells+=np.bincount(idx[valid],minlength=len(groups))
    expected={};sources={}
    for j,g in groups.iterrows():
        other='replicate_2' if g.replicate=='replicate_1' else 'replicate_1'
        key=(g.cell_line,g.drug,g.dose_label,other);source=gid.get(key)
        if source is not None:
            expected[j]=(pooled[source]+1)/(pooled[source].sum()+ng);sources[j]=source
    scored_n=np.zeros(len(groups),dtype=np.int64);dev=np.zeros(len(groups));sq=np.zeros(len(groups))
    for folder in sorted(run.glob('block_[0-9]*')):
        c=sp.load_npz(folder/'C_counts.npz').tocsr();start=int(folder.name.split('_')[1])*size;idx=labels[start:start+c.shape[0]]
        for j in np.unique(idx[idx>=0]):
            if j not in expected:continue
            sub=c[idx==j];lib=np.asarray(sub.sum(axis=1)).ravel();positive=lib>0;sub=sub[positive];lib=lib[positive]
            if not len(lib):continue
            p=expected[j];coo=sub.tocoo();v=coo.data.astype(float)
            # All-gene Poisson deviance with sum(mu)=sum(C); zero terms included analytically.
            value=2*np.sum(v*(np.log(v)-np.log(lib[coo.row])-np.log(p[coo.col])))
            y,_=normalize_A(sub);pred=np.log1p(10000*p)
            mse_total=float(y.multiply(y).sum())+len(lib)*float(pred@pred)-2*float(np.asarray(y@pred).sum())
            dev[j]+=max(float(value),0);sq[j]+=max(mse_total,0);scored_n[j]+=len(lib)
    records=[]
    for line in LINES:
        for drug in DRUGS:
            for test_rep,source_rep in [('replicate_2','replicate_1'),('replicate_1','replicate_2')]:
                ids=groups.index[(groups.cell_line==line)&(groups.drug==drug)&(groups.replicate==test_rep)].to_numpy()
                n=int(scored_n[ids].sum());total=int(n_cells[ids].sum());source_ids=[sources[j] for j in ids if j in sources]
                records.append(dict(cell_line=line,drug=drug,development_replicate=source_rep,evaluation_replicate=test_rep,
                    development_cells=int(n_cells[source_ids].sum()),evaluation_cells=total,scored_cells=n,unscored_cells=total-n,
                    n_genes=ng,n_pairs=n*ng,poisson_deviance_per_pair=float(dev[ids].sum()/(n*ng)) if n else None,
                    normalized_MSE_per_pair=float(sq[ids].sum()/(n*ng)) if n else None,
                    status='scored' if n else 'no_matched_development_stratum',
                    baseline='C-only dev pooled proportions by cell_line/drug/dose; pseudocount=1',
                    calibration='observed evaluation C library; not a molecular-holdout target',
                    sgRNA_scope='marginal baseline, no sgRNA coefficient; full labels preserved separately',
                    interpretation='exploratory cross-label sensitivity; biological independence UNRESOLVED'))
    result=pd.DataFrame(records);result.to_csv(run/'unsplit_cross_replicate_sensitivity.csv',index=False)
    return result

ENGINE_SHA = 'd3929be1d6fc1110c05daf8881e5386e91dd94d9081a3719e7818fb2242dbf27'

In [3]:
from IPython.display import display
c = np.array([0, 1, 2, 20, 1000], dtype=np.int64)
for q in [0.0, 0.5, 1.0]:
    a, b = split_values(c, q, np.random.default_rng(11))
    assert np.array_equal(a + b, c)
a, b = split_values(np.array([2, 3, 0, 7]), 0.5, np.random.default_rng(19))
ri, gi = np.array([0, 0, 1, 1]), np.array([0, 0, 1, 2])
C = sp.coo_matrix((np.array([2,3,0,7]), (ri,gi)), shape=(2,3)).tocsr()
A = sp.coo_matrix((a, (ri,gi)), shape=(2,3)).tocsr()
B = sp.coo_matrix((b, (ri,gi)), shape=(2,3)).tocsr()
assert (A+B-C).nnz == 0
Y, L_A = normalize_A(A)
assert np.allclose(expected_B_counts(Y.toarray(), L_A, 0.5), A.toarray())
assert np.allclose(expected_B_counts(Y.toarray(), L_A, 0.25), 3*A.toarray())
print('Small-input checks PASS')


Small-input checks PASS


In [4]:
RUN, split_summary = split_raw(CONFIG)
print(json.dumps(split_summary,ensure_ascii=False,indent=2))
display(pd.read_csv(RUN/'block_summary.csv').head(10))

Reuse checksum-verified completed split; no raw reread
{
  "status": "complete",
  "run_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008",
  "seed": 20261008,
  "q": 0.5,
  "algorithm": "PCG64 / NumPy binomial per positive retained coordinate record",
  "numpy": "2.4.6",
  "n_cells": 1052205,
  "n_genes": 58347,
  "n_blocks": 211,
  "A_umi": 1734209601,
  "B_umi": 1734141518,
  "C_umi": 3468351119,
  "actual_A_fraction": 0.5000098148944072,
  "A_zero_cells": 0,
  "equality": "elementwise and all-gene library A+B=C checked",
  "raw_sha256": "a1443d71dc890056bfb804d57bdef19d21942563ff088dcd1fadf08dd52f5359",
  "raw_crc": "full_stream_checked",
  "independence": "molecular_holdout; biological independence unresolved; unconditional A/B independence requires distribution assumptions",
  "population_selection": "Fixed Step 1 C>=500 and author-metadata/QC population; this conditions on C=A+B",
  "source_identity": {
    "version": "step2-count-split-

,block,rows,C_umi,A_umi,B_umi,C_nnz,A_nnz,B_nnz,A_zero_cells
0,0,5000,17774641,8886196,8888445,8784182,5181368,5183525,0
1,1,5000,17083733,8547553,8536180,8399951,4939724,4937846,0
2,2,5000,18018190,9011602,9006588,8934062,5280991,5279431,0
3,3,5000,21437729,10716958,10720771,9819376,5875646,5878274,0
4,4,5000,21391512,10697707,10693805,10236119,6125627,6121016,0
5,5,5000,19662994,9834805,9828189,9545398,5679704,5677966,0
6,6,5000,19599335,9800583,9798752,9353276,5559802,5560247,0
7,7,5000,20021592,10011831,10009761,9603106,5715584,5714282,0
8,8,5000,21850908,10924600,10926308,10217669,6135519,6135544,0
9,9,5000,20834188,10414145,10420043,10024359,5997798,5998437,0


In [5]:
preprocessing_summary = preprocess_A(CONFIG, RUN)
print(json.dumps({k:v for k,v in preprocessing_summary.items() if k!='files'},ensure_ascii=False,indent=2))
display(pd.read_parquet(RUN/'A_preprocessing/selected_genes_A.parquet').head(10))
print((RUN/'A_preprocessing/evaluation_pair_contract.json').read_text())

{
  "status": "complete",
  "fitting_scope": "all context-eligible positive-library A cells within-cell molecular holdout",
  "n_fit_cells": 989299,
  "eligible_cells": 989299,
  "evaluation_cells": 989299,
  "selected_genes": 2500,
  "requested_genes": 2500,
  "selected_PPI_edges": 10896,
  "gene_rule": "A-only logCP10K sample variance, detection fraction, gene_idx deterministic tie; unique symbols",
  "normalization": "log1p(10000*A/all-gene-A-library)",
  "zero_A_library": "retained as zero input; excluded from gene-stat fitting; no B/C rescue",
  "B_access": "none",
  "graph_metacell_model": "not fitted in Step 2; future construction/fitting/updates must use A only",
  "cross_replicate": "not development-only: do not reuse this within-cell selection for cross-replicate confirmation"
}


,gene_idx,gene_id,gene_symbol,gene_col,A_logcp10k_mean,A_logcp10k_variance,A_detection_fraction,ppi_supported,selected_col
0,4433,ENSG00000115461.4,IGFBP5,4432,1.164330,3.678150,0.308546,True,0
1,4144,ENSG00000113140.10,SPARC,4143,2.016144,3.021358,0.617334,True,1
2,9278,ENSG00000149948.13,HMGA2,9277,1.028535,2.802947,0.293490,True,2
3,11422,ENSG00000164692.17,COL1A2,11421,1.870965,2.591044,0.614544,True,3
4,17857,ENSG00000198727.2,MT-CYB,17856,2.994062,2.559284,0.814205,True,4
5,17955,ENSG00000198899.2,MT-ATP6,17954,2.785189,2.554068,0.794231,True,5
6,4424,ENSG00000115414.18,FN1,4423,1.287241,2.541280,0.427021,True,6
7,17849,ENSG00000198712.1,MT-CO2,17848,2.536056,2.519133,0.760803,True,7
8,17878,ENSG00000198763.3,MT-ND2,17877,2.488173,2.512884,0.750538,True,8
9,22127,ENSG00000212907.2,MT-ND4L,22126,2.216873,2.420635,0.705305,True,9


{
  "cells_file": "evaluation_cells.parquet",
  "cells_sha256": "0ca0d5c89eaa40aba8d78c6277a23f50c4a9b72e43bf25b695b29949af902e85",
  "genes_file": "selected_genes_A.parquet",
  "genes_sha256": "f3615c8543f6ea0b9486df589c53baed475dd516c92ffa33e61e25a97707ca3a",
  "n_cells": 989299,
  "n_genes": 2500,
  "n_pairs": 2473247500,
  "pair_rule": "all selected genes, including observed zeros, for context-eligible positive-A-library cells; frozen without B access",
  "q": 0.5,
  "normalization_scale": 10000,
  "method_comparison": "all methods must use exactly these same cell-gene pairs"
}


In [6]:
sensitivity = unsplit_sensitivity(RUN)
display(sensitivity[['cell_line','drug','development_replicate','evaluation_replicate','scored_cells','unscored_cells','poisson_deviance_per_pair','normalized_MSE_per_pair','status']])

,cell_line,drug,development_replicate,evaluation_replicate,scored_cells,unscored_cells,poisson_deviance_per_pair,normalized_MSE_per_pair,status
0,A172,Vehicle,replicate_1,replicate_2,32191,0,0.138576,0.067778,scored
1,A172,Vehicle,replicate_2,replicate_1,33460,0,0.144552,0.067318,scored
2,A172,Lapatinib,replicate_1,replicate_2,48061,0,0.139724,0.066995,scored
3,A172,Lapatinib,replicate_2,replicate_1,47258,0,0.145050,0.067005,scored
4,A172,Nintedanib,replicate_1,replicate_2,39073,0,0.118397,0.063608,scored
5,A172,Nintedanib,replicate_2,replicate_1,36307,0,0.135199,0.060088,scored
6,A172,Trametinib,replicate_1,replicate_2,19639,0,0.131357,0.058999,scored
7,A172,Trametinib,replicate_2,replicate_1,13723,0,0.151684,0.055149,scored
8,A172,ZSTK474,replicate_1,replicate_2,30815,0,0.098427,0.070978,scored
9,A172,ZSTK474,replicate_2,replicate_1,34561,0,0.120193,0.065785,scored


In [7]:
context_totals = save_protocol(CONFIG, RUN, split_summary, preprocessing_summary)
display(context_totals)
display(pd.read_csv(RUN/'information_flow.csv'))
print('Step 2 outputs:', RUN)
print('Summary:', ROOT/'results/raw/step2_summary.json')

,cell_line,drug,n_cells
0,A172,Vehicle,65651
1,A172,Lapatinib,95319
2,A172,Nintedanib,75380
3,A172,Trametinib,33362
4,A172,ZSTK474,65376
5,T98G,Vehicle,70400
6,T98G,Lapatinib,102647
7,T98G,Nintedanib,49071
8,T98G,Trametinib,50865
9,T98G,ZSTK474,52472


,stage,allowed_information,status
0,cell_population,Step 1 C>=500 and author QC fixed,selection conditioning explicitly recorded
1,normalization/gene_selection,A only,implemented
2,metacell/cell_graph/estimator/adaptive_updates,A only,future fitting not performed here
3,B aggregation,fixed A-only assignment,never regroup using B
4,hyperparameter_selection,pre-fixed settings or separate development tar...,final B excluded
5,final_evaluation,B only after freeze,no B-based preprocessing/calibration
6,count_calibration,A library and q only,fixed expected_B_counts function
7,cross_replicate_sensitivity,C unsplit; dev-only preprocessing in each dire...,batch/label sensitivity; independent biologica...


Step 2 outputs: /home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008
Summary: /home/featurize/work/biological_prior_validation/results/raw/step2_summary.json
